# Adaptive Artificial Nervous System: CPU quickstart
Upload the project ZIP, then run the original experiment and exploratory guard on fresh outputs. No GPU/API key needed. The Colab interface itself has not been tested; the equivalent subprocess commands passed locally.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, tempfile, sys, subprocess
uploaded = files.upload()
names = [n for n in uploaded if n.endswith('.zip')]
assert len(names) == 1, 'Upload Adaptive_Nervous_System.zip only'
root = Path(tempfile.mkdtemp(prefix='ans_lab_')).resolve()
with zipfile.ZipFile(names[0]) as z:
    for info in z.infolist():
        if not (root/info.filename).resolve().is_relative_to(root):
            raise ValueError('Unsafe ZIP path')
    z.extractall(root)
project = root/'Adaptive_Nervous_System'
assert (project/'run_lab.py').is_file()
print(project)

In [ ]:
def run(*args):
    subprocess.run([sys.executable,*args],cwd=project,check=True)
run('-m','pip','install','-r','requirements.txt')
run('-m','unittest','-v','test_lab')

In [ ]:
run('run_lab.py','--out','fresh_results')
run('followup.py','--out','fresh_results/followup')

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
primary = json.loads((project/'fresh_results/summary.json').read_text())
followup = json.loads((project/'fresh_results/followup/summary.json').read_text())
for title, data in [('Primary experiment',primary),('Exploratory follow-up',followup)]:
    print(title, 'episodes:',data['episodes'])
    display(pd.DataFrame(data['overall']).T.round(2))
    pd.Series({k:v['mean_backlog'] for k,v in data['overall'].items()}).plot.bar(title=title+' (lower is better)',ylabel='Mean backlog',rot=0)
    plt.tight_layout();plt.show()

In [ ]:
import shutil
archive = shutil.make_archive(str(root/'fresh_results'),'zip',project/'fresh_results')
files.download(archive)